# Analyze One IRMAS Test Run

This notebook evaluates one single-label IRMAS Mel run on `data/IRMAS/IRMAS-TestingData-Part1/Part1` and exposes the detailed result tables for inspection.

For bulk model-to-model comparison in DataFrames, use `src/test/compare_tests.ipynb`.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython import get_ipython
from IPython.display import display

from src.test.evaluate import evaluate_irmas_part1_run
from src.test.utils import (
    build_single_label_confusion_matrix_figure,
    build_single_label_roc_curves_figure,
    build_training_accuracy_figure,
    build_training_loss_figure,
    load_training_history_frame,
)
from src.train.run_train import _repo_root

ip = get_ipython()
if ip is not None:
    ip.run_line_magic("matplotlib", "inline")

PROJECT_ROOT = _repo_root()
TEST_ROOT = PROJECT_ROOT / "data" / "IRMAS" / "IRMAS-TestingData-Part1" / "Part1"
CACHE_ROOT = PROJECT_ROOT / "data" / "processed" / "irmas_test_part1_mels"


MODEL_PATH = PROJECT_ROOT / "src" / "models" / "saved_weights" / "irmas_single_label_mel"
FORCE_REBUILD_FEATURES = False

BATCH_SIZE = 32
NUM_WORKERS = 0

print(f"Project root: {PROJECT_ROOT}")
print(f"IRMAS test root: {TEST_ROOT}")
print(f"Model path: {MODEL_PATH}")


In [ ]:
result = evaluate_irmas_part1_run(
    MODEL_PATH,
    test_root=TEST_ROOT,
    cache_root=CACHE_ROOT,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    force_rebuild_features=FORCE_REBUILD_FEATURES,
    save_artifacts=False,
)

summary_df = pd.DataFrame([result["summary_row"]])
per_class_df = result["report"]["per_class"].sort_values(
    ["f1", "support"],
    ascending=[False, False],
).reset_index(drop=True)
samples_df = result["samples_df"].copy()
errors_df = samples_df.loc[~samples_df["primary_match"]].reset_index(drop=True)

display(summary_df)
print(f"Evaluation samples: {len(samples_df)}")
print(f"Misclassified clips: {len(errors_df)}")


## Training History

These cells load the training history stored inside the checkpoint and plot convergence across epochs.


In [ ]:
history_df, history_meta = load_training_history_frame(
    MODEL_PATH,
    project_root=PROJECT_ROOT,
)

history_summary_df = pd.DataFrame([history_meta])
best_epoch_df = (
    history_df.loc[[history_df["val_loss"].idxmin()]].reset_index(drop=True)
    if "val_loss" in history_df.columns and history_df["val_loss"].notna().any()
    else history_df.tail(1).reset_index(drop=True)
)

display(history_summary_df)
display(best_epoch_df)
display(history_df)


In [ ]:
loss_fig = build_training_loss_figure(history_df)
display(loss_fig)


In [ ]:
accuracy_fig = build_training_accuracy_figure(history_df)
display(accuracy_fig)


In [ ]:
display(per_class_df)


display(
    errors_df.groupby(["primary_true_label", "predicted_label"])
    .size()
    .rename("count")
    .reset_index()
    .sort_values("count", ascending=False)
    .head(20)
)


## Plots

Run the cells below after evaluation to render the confusion matrix and ROC curves separately.


In [ ]:
confusion_fig = build_single_label_confusion_matrix_figure(
    result["outputs"]["y_true"],
    result["outputs"]["y_pred"],
    result["run"].classes,
)

display(confusion_fig)


In [ ]:
roc_fig = build_single_label_roc_curves_figure(
    result["outputs"]["y_true"],
    result["outputs"]["y_prob"],
    result["run"].classes,
)

display(roc_fig)
